# ♻️ Classificador de resíduos recicláveis

Treina uma rede neural capaz de identificar o **tipo de material** de um resíduo
a partir de uma foto — papelão, vidro, metal, papel, plástico — e exporta o
modelo em **ONNX** para rodar direto no navegador.

**Framework:** Keras / TensorFlow
**Dataset:** Roboflow Universe (classificação em pastas)
**Modelo:** MobileNetV2 com transfer learning
**Saída:** `modelo.onnx` + `metadados.json`, prontos para o site

---

## ⚙️ Antes de começar

**Ative a GPU:** `Ambiente de execução → Alterar o tipo de ambiente de execução
→ Acelerador de hardware: GPU (T4) → Salvar`

**Tempo estimado:** 10 a 15 minutos.

**Duas coisas pedem a sua mão:** a chave da API do Roboflow e o trecho de
download do dataset escolhido (etapa 2). O resto roda sozinho.

---

## Por que este projeto termina em ONNX

O objetivo não é só treinar: é **colocar o modelo para rodar num site**. O ONNX
é um formato aberto que o navegador executa via `onnxruntime-web`, sem servidor,
sem Python e sem enviar a foto do usuário para lugar nenhum — tudo acontece na
máquina de quem acessa.

Isso impõe três decisões que aparecem ao longo do notebook:

| Decisão | Motivo |
|---|---|
| **MobileNetV2** como base | Arquitetura desenhada para dispositivos leves; o ONNX sai com cerca de 9 MB |
| **Pré-processamento dentro do modelo** | O JavaScript manda os pixels crus de 0 a 255; o modelo se vira com o resto |
| **Exportar `metadados.json`** | O site precisa saber os nomes dos tensores, o tamanho da imagem e os nomes das classes |

---

## Roteiro

| Etapa | O que acontece |
|---|---|
| 1 | Prepara o ambiente e confirma a GPU |
| 2 | Baixa o dataset do Roboflow |
| 3 | Explora as classes e o balanceamento |
| 4 | Monta os pipelines de dados com augmentation |
| 5 | Constrói o modelo (MobileNetV2 + cabeça) |
| 6 | Treina em duas fases: base congelada e ajuste fino |
| 7 | Avalia com matriz de confusão e métricas por classe |
| 8 | **Exporta para ONNX e confere se bate com o Keras** |
| 9 | Gera os arquivos para o site e baixa tudo |

---
# 1. Preparação

Instalamos o `roboflow` (download do dataset) e o trio `tf2onnx` / `onnx` /
`onnxruntime` (conversão e verificação do modelo exportado). O TensorFlow já
vem no Colab.

> ⚠️ **O pip vai imprimir um bloco vermelho de conflito. Isso é esperado.**
>
> O `onnx` exige uma versão de protobuf mais nova que a de três pacotes
> pré-instalados no Colab: `google-ai-generativelanguage`, `ydf` e
> `grpcio-status`. Nenhum dos três é usado aqui.
>
> Repare que o **TensorFlow não aparece na lista de conflitos** — ele aceita a
> versão nova do protobuf sem problema. A instalação foi concluída; é um aviso,
> não uma falha. A célula seguinte confirma isso importando tudo de fato.

In [ ]:
!pip install -q roboflow tf2onnx onnx onnxruntime

In [ ]:
import importlib

print("Conferindo se tudo importa de verdade:\n")

pacotes = [
    ("tensorflow", "TensorFlow"),
    ("keras", "Keras"),
    ("onnx", "onnx"),
    ("onnxruntime", "onnxruntime"),
    ("tf2onnx", "tf2onnx"),
    ("roboflow", "roboflow"),
    ("google.protobuf", "protobuf"),
]

houve_falha = False
for modulo, rotulo in pacotes:
    try:
        m = importlib.import_module(modulo)
        print(f"  [ok] {rotulo:<14} {getattr(m, '__version__', '?')}")
    except Exception as erro:
        print(f"  [!!] {rotulo:<14} FALHOU: {erro}")
        houve_falha = True

if houve_falha:
    print("\n[!] Algum pacote não importou.")
    print("    Ambiente de execução > Reiniciar sessão, e rode de novo.")
else:
    print("\nTudo certo — os avisos vermelhos do pip podem ser ignorados.")

In [ ]:
import random

import numpy as np
import tensorflow as tf
from tensorflow import keras

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
tf.random.set_seed(SEMENTE)
keras.utils.set_random_seed(SEMENTE)

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU        : {gpus[0].name}")
    print("\nTudo certo, pode seguir.")
else:
    print("GPU        : nenhuma detectada")
    print("\n[!] Sem GPU o treino fica muito lento.")
    print("    Ambiente de execução > Alterar o tipo de ambiente de execução")
    print("    > Acelerador de hardware: GPU (T4) > Salvar")

> 💡 **Se o ambiente reiniciar**, as variáveis se perdem e aparecem erros do
> tipo `NameError`. Cada célula deste notebook importa o que usa, então os
> imports se resolvem sozinhos — mas os **dados** e o **modelo** precisam ser
> recarregados. Rode de novo a partir da etapa 2.

---
# 2. Baixando o dataset do Roboflow

## Como escolher e exportar

1. Vá em [universe.roboflow.com](https://universe.roboflow.com) e busque por
   `waste classification` ou `recycling classification`
2. **Confira que o projeto é de _Classification_, não de _Object Detection_.**
   Boa parte dos resultados da busca é detecção, e detecção não serve aqui
3. Abra o dataset → **Download Dataset** → Format: **Folder Structure** →
   **show download code**
4. Cole o trecho gerado na célula abaixo

## Três datasets já verificados

Para você não perder tempo na busca, estes são projetos de **classificação**
(não de detecção) que encontrei no Universe, com o identificador pronto:

| Workspace / Projeto | Imagens | Classes |
|---|---|---|
| `garbage-vhmrm` / `garbage-hforb` | ~2.500 | cardboard, glass, metal, paper, plastic, trash |
| `teamproject-3f4si` / `garbage-pqluq` | ~2.400 | cardboard, glass, metal, paper, plastic |
| `trashclassification-tayqe` / `trash-classification-vky3a` | ~1.600 | 8 classes, inclui compost e recycle |

O primeiro é o mais equilibrado: seis classes claras e tamanho bom para treinar
em poucos minutos. A célula abaixo já vem preenchida com ele — basta colocar sua
chave da API.

A chave sai de **app.roboflow.com → Settings → API Keys**.

> Se preferir escolher outro, o filtro que mais importa na busca é o tipo do
> projeto: precisa ser **Classification**. Boa parte dos resultados de "waste"
> no Universe é detecção, e detecção não serve aqui.

## Por que "Folder Structure"

Esse formato entrega as imagens organizadas assim:

```
dataset/
├── train/
│   ├── cardboard/   ├── glass/   ├── metal/   ├── paper/   └── plastic/
├── valid/
│   └── (as mesmas pastas)
└── test/
    └── (as mesmas pastas)
```

**O nome da pasta é o rótulo.** É exatamente o que a função
`image_dataset_from_directory` do Keras espera, então não precisamos escrever
nenhum código de leitura de rótulo.

In [ ]:
from roboflow import Roboflow

# ↓↓↓ COLOQUE SUA CHAVE DA API AQUI ↓↓↓
rf = Roboflow(api_key="SUA_API_KEY")

# Já preenchido com um dataset de classificação verificado.
# Para usar outro, troque workspace/projeto ou cole o trecho que o
# Roboflow gera em "Download Dataset > Folder Structure > show download code".
projeto = rf.workspace("garbage-vhmrm").project("garbage-hforb")
versao = projeto.version(1)

# "folder" é o formato de classificação (pastas por classe)
dataset = versao.download("folder", location="dados")

print("\nDataset baixado em:", dataset.location)

### Onde os arquivos foram parar

Em vez de assumir que o dataset está em `dados/train`, perguntamos ao próprio
objeto devolvido pelo Roboflow. A estrutura pode variar conforme a versão do
pacote, e caminho escrito à mão quebra sem aviso.

In [ ]:
from pathlib import Path

RAIZ = Path(dataset.location)

# O Roboflow usa "valid", mas alguns exports usam "val"
def achar_divisao(raiz, nomes):
    for nome in nomes:
        caminho = raiz / nome
        if caminho.is_dir():
            return caminho
    return None

DIR_TREINO = achar_divisao(RAIZ, ["train"])
DIR_VALID = achar_divisao(RAIZ, ["valid", "val"])
DIR_TESTE = achar_divisao(RAIZ, ["test"])

print("Raiz      :", RAIZ)
print("Treino    :", DIR_TREINO)
print("Validação :", DIR_VALID)
print("Teste     :", DIR_TESTE or "(não existe — usaremos a validação)")

assert DIR_TREINO is not None, "pasta de treino não encontrada"
assert DIR_VALID is not None, "pasta de validação não encontrada"

---
# 3. Explorando o dataset

Duas perguntas antes de treinar:

1. **Quais classes existem de fato?** O nome das pastas é a fonte da verdade —
   não o que o site do Roboflow anunciava.
2. **Estão balanceadas?** Uma classe com dez vezes menos exemplos vai ser
   sistematicamente ignorada pelo modelo, e a acurácia média esconde isso.

In [ ]:
from pathlib import Path

EXTENSOES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def contar_por_classe(pasta):
    """Conta imagens em cada subpasta (cada subpasta é uma classe)."""
    contagem = {}
    for sub in sorted(p for p in pasta.iterdir() if p.is_dir()):
        n = sum(1 for a in sub.rglob("*") if a.suffix.lower() in EXTENSOES)
        contagem[sub.name] = n
    return contagem


treino = contar_por_classe(DIR_TREINO)
valid = contar_por_classe(DIR_VALID)
teste = contar_por_classe(DIR_TESTE) if DIR_TESTE else {}

CLASSES = sorted(treino)
print(f"{len(CLASSES)} classes encontradas: {CLASSES}\n")

print(f"{'classe':<20}{'treino':>9}{'valid':>9}{'teste':>9}{'total':>9}")
print("-" * 56)
for c in CLASSES:
    t, v, te = treino.get(c, 0), valid.get(c, 0), teste.get(c, 0)
    print(f"{c:<20}{t:>9}{v:>9}{te:>9}{t+v+te:>9}")
print("-" * 56)
print(f"{'TOTAL':<20}{sum(treino.values()):>9}{sum(valid.values()):>9}"
      f"{sum(teste.values()):>9}{sum(treino.values())+sum(valid.values())+sum(teste.values()):>9}")

# Diagnóstico de balanceamento
valores = [v for v in treino.values() if v > 0]
razao = max(valores) / min(valores)
print()
if razao > 5:
    print(f"[!] Desbalanceamento forte ({razao:.1f}x entre a maior e a menor classe).")
    print("    Acompanhe o desempenho POR CLASSE na etapa 7, não só a média.")
elif razao > 2:
    print(f"[~] Desbalanceamento moderado ({razao:.1f}x). Aceitável.")
else:
    print(f"[ok] Classes bem balanceadas ({razao:.1f}x).")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

EXTENSOES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Com muitas classes a figura ficaria alta demais: mostramos até 8
MOSTRAR = CLASSES[:8]
if len(CLASSES) > 8:
    print(f"Exibindo 8 das {len(CLASSES)} classes.\n")

fig, axes = plt.subplots(len(MOSTRAR), 4, figsize=(11, 2.6 * len(MOSTRAR)))
axes = axes.reshape(len(MOSTRAR), 4)

for linha, classe in enumerate(MOSTRAR):
    pasta = DIR_TREINO / classe
    arquivos = sorted(a for a in pasta.rglob("*")
                      if a.suffix.lower() in EXTENSOES)[:4]
    for coluna in range(4):
        ax = axes[linha, coluna]
        ax.axis("off")
        if coluna < len(arquivos):
            ax.imshow(Image.open(arquivos[coluna]).convert("RGB"))
            if coluna == 0:
                ax.set_title(classe, fontsize=11, fontweight="bold", loc="left")

fig.suptitle("Amostras de cada classe", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
# 4. Pipelines de dados

## O tamanho da imagem: 224×224

Não é escolha arbitrária. É a resolução em que o MobileNetV2 foi treinado no
ImageNet — usar a mesma aproveita melhor os pesos pré-treinados.

## Augmentation fora do modelo

No projeto do CIFAR-100 as camadas de augmentation ficavam **dentro** do
modelo. Aqui elas ficam no **pipeline de dados**, e a razão é específica deste
projeto: o modelo vai ser exportado para ONNX, e camadas aleatórias no grafo
exportado geram operações estranhas ou simplesmente falham na conversão.

Mantendo o modelo "puro" — só pré-processamento e convoluções — o ONNX sai
limpo e previsível.

## O que fazemos com os dados

| Operação | Onde | Por quê |
|---|---|---|
| `cache()` | treino e validação | Evita reler o disco a cada época |
| `shuffle()` | só treino | A ordem não pode virar um padrão que o modelo decore |
| `prefetch()` | treino e validação | A CPU prepara o próximo lote enquanto a GPU calcula o atual |
| Augmentation | só treino | Na validação queremos medir o desempenho real, sem ruído |

Note que **não normalizamos aqui**. A normalização vai dentro do modelo, na
etapa 5 — é o que permite o navegador mandar pixels crus depois.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

TAMANHO = 224
LOTE = 32
AUTO = tf.data.AUTOTUNE

def carregar(pasta, embaralhar):
    return keras.utils.image_dataset_from_directory(
        pasta,
        labels="inferred",          # o nome da pasta é o rótulo
        label_mode="int",
        class_names=CLASSES,        # ordem fixa: o site depende dela
        image_size=(TAMANHO, TAMANHO),
        batch_size=LOTE,
        shuffle=embaralhar,
        seed=SEMENTE,
    )

ds_treino = carregar(DIR_TREINO, True)
ds_valid = carregar(DIR_VALID, False)
ds_teste = carregar(DIR_TESTE, False) if DIR_TESTE else ds_valid

# Augmentation aplicada só no treino, dentro do pipeline
aumentar = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="aumento")

ds_treino = (ds_treino
             .cache()
             .shuffle(500, seed=SEMENTE)
             .map(lambda x, y: (aumentar(x, training=True), y),
                  num_parallel_calls=AUTO)
             .prefetch(AUTO))
ds_valid = ds_valid.cache().prefetch(AUTO)
ds_teste = ds_teste.cache().prefetch(AUTO)

print(f"\nOrdem das classes (importa para o site): {CLASSES}")
for imagens, rotulos in ds_treino.take(1):
    print(f"Lote de imagens: {imagens.shape}, {imagens.dtype}")
    print(f"Faixa de pixels: {float(tf.reduce_min(imagens)):.1f} "
          f"a {float(tf.reduce_max(imagens)):.1f}")
    print(f"Lote de rótulos: {rotulos.shape}, exemplos {rotulos[:8].numpy()}")

---
# 5. O modelo

## Transfer learning, desta vez sim

No projeto do CIFAR-100 treinamos do zero, porque as imagens tinham 32×32 e não
havia o que aproveitar de um modelo treinado em 224×224. Aqui a situação se
inverte: as imagens do Roboflow são grandes, e o MobileNetV2 já viu 1,2 milhão
de fotos do ImageNet — incluindo garrafas, latas e caixas de papelão.

Ele já sabe reconhecer bordas, texturas, brilho de metal e transparência de
vidro. Só precisamos ensinar a decisão final.

## A estrutura

```
Entrada (224, 224, 3), pixels de 0 a 255
      ↓
Rescaling(1/127.5, offset=-1)     normaliza para [-1, 1]
      ↓
MobileNetV2 (pesos do ImageNet, sem a camada final)
      ↓
GlobalAveragePooling2D            1280 características
      ↓
Dropout(0.3)
      ↓
Dense(n_classes, softmax)         probabilidades
```

## A decisão mais importante do projeto

A camada `Rescaling` fica **dentro do modelo**. Poderíamos normalizar os pixels
no pipeline de dados e deixar o modelo esperando valores já em `[-1, 1]` — é o
mais comum em tutoriais.

Mas então o JavaScript do site teria que reproduzir exatamente essa conta antes
de chamar o modelo. Qualquer divergência — dividir por 255 em vez de 127.5,
esquecer o deslocamento — produz predições silenciosamente erradas. É um dos
bugs mais difíceis de achar em aplicação de visão no navegador, porque nada
quebra: o modelo só passa a errar.

Colocando a normalização dentro do grafo, **o contrato com o site fica
trivial**: mande os pixels de 0 a 255, receba probabilidades. Impossível errar.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def construir_modelo(n_classes, tamanho=224):
    base = keras.applications.MobileNetV2(
        input_shape=(tamanho, tamanho, 3),
        include_top=False,           # descartamos a cabeça de 1000 classes
        weights="imagenet",
    )
    base.trainable = False           # fase 1: base congelada

    entrada = keras.Input(shape=(tamanho, tamanho, 3), name="imagem")

    # Normalização DENTRO do modelo: o site manda pixels crus de 0 a 255
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="normalizacao")(entrada)

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="pooling")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    saida = layers.Dense(n_classes, activation="softmax",
                         name="probabilidades")(x)

    modelo = keras.Model(entrada, saida, name="classificador_residuos")
    return modelo, base


modelo, base_mobilenet = construir_modelo(len(CLASSES), TAMANHO)

treinaveis = sum(1 for c in modelo.layers if c.trainable)
print(f"Parâmetros totais ..... {modelo.count_params():,}")
print(f"Parâmetros treináveis . {sum(int(tf.size(w)) for w in modelo.trainable_weights):,}")
print(f"\nEntrada: {modelo.input_shape}  ->  Saída: {modelo.output_shape}")
print(f"Classes: {CLASSES}")

---
# 6. Treino em duas fases

Transfer learning bem feito tem duas etapas, e a ordem importa.

## Fase 1 — base congelada

Só a cabeça de classificação aprende. A base do MobileNetV2 fica intacta.

O motivo é sutil e frequentemente ignorado: a cabeça começa com **pesos
aleatórios**, então os primeiros gradientes são enormes e caóticos. Se a base
estivesse destravada, esses gradientes destruiriam as features do ImageNet logo
nas primeiras iterações — exatamente o que viemos aproveitar.

## Fase 2 — ajuste fino

Com a cabeça já calibrada, destravamos as camadas finais da base e treinamos
com uma taxa de aprendizado **dez vezes menor**. Agora a rede pode adaptar suas
features de alto nível ao nosso domínio (garrafas, latas, papelão) sem estragar
as de baixo nível (bordas, texturas), que são universais.

Destravamos só as últimas camadas: as iniciais detectam bordas e cores, que
valem igual para qualquer imagem, e retreiná-las seria desperdício e risco.

In [ ]:
from tensorflow import keras

modelo.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_fase1 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                  restore_best_weights=True, verbose=1),
]

print("FASE 1 — base congelada, só a cabeça aprende\n")
historico1 = modelo.fit(
    ds_treino,
    validation_data=ds_valid,
    epochs=15,
    callbacks=callbacks_fase1,
    verbose=2,
)

In [ ]:
from tensorflow import keras

# Destrava as últimas camadas da base
base_mobilenet.trainable = True
CAMADAS_CONGELADAS = 100
for camada in base_mobilenet.layers[:CAMADAS_CONGELADAS]:
    camada.trainable = False

# BatchNorm continua congelada: com lote pequeno, atualizar suas estatísticas
# desestabiliza o ajuste fino. É a pegadinha clássica de fine-tuning.
for camada in base_mobilenet.layers:
    if isinstance(camada, keras.layers.BatchNormalization):
        camada.trainable = False

# Recompilar é OBRIGATÓRIO: sem isso o Keras mantém o grafo antigo e o
# destravamento não tem efeito nenhum.
modelo.compile(
    optimizer=keras.optimizers.Adam(1e-4),     # 10x menor que a fase 1
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print(f"Camadas destravadas: {len(base_mobilenet.layers) - CAMADAS_CONGELADAS} "
      f"de {len(base_mobilenet.layers)}")
print(f"Parâmetros treináveis: "
      f"{sum(int(tf.size(w)) for w in modelo.trainable_weights):,}\n")

callbacks_fase2 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=3, min_lr=1e-7, verbose=1),
]

print("FASE 2 — ajuste fino das camadas finais\n")
historico2 = modelo.fit(
    ds_treino,
    validation_data=ds_valid,
    epochs=15,
    callbacks=callbacks_fase2,
    verbose=2,
)

In [ ]:
import matplotlib.pyplot as plt

acc = historico1.history["accuracy"] + historico2.history["accuracy"]
val_acc = historico1.history["val_accuracy"] + historico2.history["val_accuracy"]
perda = historico1.history["loss"] + historico2.history["loss"]
val_perda = historico1.history["val_loss"] + historico2.history["val_loss"]
corte = len(historico1.history["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

for ax, (tr, va, titulo) in zip(
        axes, [(acc, val_acc, "Acurácia"), (perda, val_perda, "Perda")]):
    ax.plot(tr, label="treino", color="#2E86AB")
    ax.plot(va, label="validação", color="#C0392B")
    ax.axvline(corte - 0.5, color="gray", linestyle="--", linewidth=1.4)
    ax.text(corte - 0.4, ax.get_ylim()[0], " ajuste fino →",
            fontsize=9, color="gray", va="bottom")
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("época")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Melhor acurácia de validação: {max(val_acc)*100:.2f}%")
print(f"  ao final da fase 1: {max(historico1.history['val_accuracy'])*100:.2f}%")
print(f"  ao final da fase 2: {max(historico2.history['val_accuracy'])*100:.2f}%")

---
# 7. Avaliação

Medindo no conjunto de teste — imagens que não participaram nem do treino nem
da validação.

In [ ]:
import numpy as np

resultado = modelo.evaluate(ds_teste, verbose=0, return_dict=True)

print("=" * 48)
print("DESEMPENHO NO CONJUNTO DE TESTE")
print("=" * 48)
print(f"  Perda .... {resultado['loss']:.4f}")
print(f"  Acurácia . {resultado['accuracy']*100:.2f}%")
print(f"\n  Acerto aleatório seria {100/len(CLASSES):.1f}%")
print("=" * 48)

# Predições para as análises seguintes
y_prob = modelo.predict(ds_teste, verbose=0)
y_pred = y_prob.argmax(axis=1)
y_real = np.concatenate([r.numpy() for _, r in ds_teste])

print(f"\nAcertos: {(y_pred == y_real).sum()} de {len(y_real)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

print(classification_report(y_real, y_pred, target_names=CLASSES,
                            digits=3, zero_division=0))

mc = confusion_matrix(y_real, y_pred, normalize="true")
lado = max(7, 0.62 * len(CLASSES))
fig, ax = plt.subplots(figsize=(lado, lado * 0.88))
ConfusionMatrixDisplay(mc, display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_title("Matriz de confusão (normalizada por linha)", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Como ler a matriz.** Cada linha é a classe verdadeira, cada coluna é o
palpite do modelo. A diagonal deve concentrar os valores.

Vale olhar especificamente os pares que costumam se confundir em resíduos:
vidro com plástico transparente (ambos translúcidos e brilhantes) e papel com
papelão (mesma cor, diferença só de textura e espessura). Se a confusão do seu
modelo estiver aí, é um erro compreensível. Se estiver entre metal e papel, tem
algo errado nos dados.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Os erros em que o modelo estava MAIS confiante — os mais instrutivos
erros = np.where(y_pred != y_real)[0]

if len(erros) == 0:
    print("Nenhum erro no conjunto de teste.")
else:
    confianca = y_prob[erros, y_pred[erros]]
    piores = erros[np.argsort(-confianca)][:8]

    imagens = np.concatenate([i.numpy() for i, _ in ds_teste])

    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    for ax, i in zip(axes.ravel(), piores):
        ax.imshow(imagens[i].astype("uint8"))
        ax.axis("off")
        ax.set_title(f"real: {CLASSES[y_real[i]]}\n"
                     f"previu: {CLASSES[y_pred[i]]} "
                     f"({y_prob[i, y_pred[i]]*100:.0f}%)",
                     fontsize=9, color="#C0392B")
    for ax in axes.ravel()[len(piores):]:
        ax.axis("off")

    fig.suptitle("Os erros mais confiantes — onde o modelo errou sem hesitar",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
# 8. Exportação para ONNX

Chegamos ao objetivo do projeto.

## Por que não dá para converter direto

O `tf2onnx` não aceita um modelo Keras 3 como entrada — a API mudou e o
conversor não acompanhou. O caminho que funciona tem duas etapas:

```
Modelo Keras 3  →  SavedModel (model.export)  →  ONNX (tf2onnx)
```

O `model.export()` é diferente do `model.save()`: ele grava o modelo no formato
SavedModel do TensorFlow, **já em modo de inferência**, congelando o
comportamento das camadas que se comportam diferente no treino.

## A verificação que ninguém faz

Converter é fácil; converter **certo** é outra coisa. Depois da conversão,
rodamos as mesmas imagens nos dois modelos e comparamos as saídas número a
número. Se a diferença for maior que um erro de arredondamento, algo se perdeu
no caminho — e é muito melhor descobrir aqui do que depois, com o site
retornando resultados errados sem nenhuma mensagem de erro.

In [ ]:
import subprocess
import sys
from pathlib import Path

DIR_SAVEDMODEL = "saved_model"
ARQUIVO_ONNX = "modelo.onnx"

# Etapa 1: Keras 3 -> SavedModel, já em modo de inferência
modelo.export(DIR_SAVEDMODEL)
print("SavedModel gerado:", sorted(p.name for p in Path(DIR_SAVEDMODEL).iterdir()))

# Etapa 2: SavedModel -> ONNX
processo = subprocess.run(
    [sys.executable, "-m", "tf2onnx.convert",
     "--saved-model", DIR_SAVEDMODEL,
     "--output", ARQUIVO_ONNX,
     "--opset", "13"],
    capture_output=True, text=True,
)

if processo.returncode != 0:
    print("[!] A conversão falhou:")
    print(processo.stderr[-3000:])
else:
    tamanho = Path(ARQUIVO_ONNX).stat().st_size / 1e6
    print(f"\nONNX gerado: {ARQUIVO_ONNX} ({tamanho:.1f} MB)")

In [ ]:
import numpy as np
import onnx
import onnxruntime as ort

# O modelo é estruturalmente válido?
grafo = onnx.load(ARQUIVO_ONNX)
onnx.checker.check_model(grafo)
print("onnx.checker: modelo válido")

sessao = ort.InferenceSession(ARQUIVO_ONNX, providers=["CPUExecutionProvider"])

NOME_ENTRADA = sessao.get_inputs()[0].name
NOME_SAIDA = sessao.get_outputs()[0].name

print("\nContrato do modelo (o site precisa disto):")
for e in sessao.get_inputs():
    print(f"  entrada : '{e.name}'  formato {e.shape}  tipo {e.type}")
for s in sessao.get_outputs():
    print(f"  saída   : '{s.name}'  formato {s.shape}  tipo {s.type}")

In [ ]:
import numpy as np

# A verificação decisiva: Keras e ONNX concordam?
lote_imagens, _ = next(iter(ds_teste))
amostra = lote_imagens.numpy().astype(np.float32)

saida_keras = modelo.predict(amostra, verbose=0)
saida_onnx = sessao.run(None, {NOME_ENTRADA: amostra})[0]

diferenca = np.abs(saida_keras - saida_onnx).max()
concordam = (saida_keras.argmax(1) == saida_onnx.argmax(1)).mean()

print("=" * 52)
print("KERAS  x  ONNX")
print("=" * 52)
print(f"  Diferença máxima nas probabilidades . {diferenca:.2e}")
print(f"  Mesma classe prevista ............... {concordam*100:.1f}%")
print("=" * 52)

if diferenca < 1e-4 and concordam == 1.0:
    print("\n[ok] Conversão fiel. O ONNX pode ir para o site.")
else:
    print("\n[!] As saídas divergiram. NÃO use este ONNX antes de investigar.")

---
# 9. Os arquivos para o site

O ONNX sozinho não basta. Ele devolve um vetor de números — nada nele diz que a
posição 3 significa "plástico", nem qual o tamanho de imagem esperado.

Geramos então um `metadados.json` com o **contrato completo** do modelo. Esse
arquivo é o que permite ao site funcionar sem nenhum valor escrito à mão no
código JavaScript — e é também o que torna possível aceitar o modelo ONNX de
outra pessoa depois.

In [ ]:
import json
from pathlib import Path

metadados = {
    "nome_modelo": "classificador_residuos",
    "descricao": "Classifica o material de um resíduo a partir de uma foto",
    "classes": CLASSES,
    "n_classes": len(CLASSES),
    "entrada": {
        "nome": NOME_ENTRADA,
        "formato": "NHWC",
        "altura": TAMANHO,
        "largura": TAMANHO,
        "canais": 3,
        "tipo": "float32",
        "faixa_pixels": "0-255",
        "observacao": ("A normalização está DENTRO do modelo. Envie os pixels "
                       "crus de 0 a 255, sem dividir por 255 nem subtrair média."),
    },
    "saida": {
        "nome": NOME_SAIDA,
        "formato": "[lote, n_classes]",
        "tipo": "float32",
        "ativacao": "softmax",
        "observacao": "Probabilidades já somando 1. Não aplique softmax de novo.",
    },
    "metricas": {
        "acuracia_teste": round(float(resultado["accuracy"]), 4),
        "perda_teste": round(float(resultado["loss"]), 4),
    },
    "treino": {
        "arquitetura_base": "MobileNetV2",
        "pesos_iniciais": "imagenet",
        "estrategia": "transfer learning em duas fases",
    },
}

Path("metadados.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(metadados, indent=2, ensure_ascii=False))

In [ ]:
from pathlib import Path

# O modelo em formato Keras também, caso você queira retomar o treino depois
modelo.save("modelo.keras")

print("Arquivos gerados:\n")
for nome in ["modelo.onnx", "metadados.json", "modelo.keras"]:
    caminho = Path(nome)
    if caminho.exists():
        print(f"  {nome:<20} {caminho.stat().st_size/1e6:7.2f} MB")
    else:
        print(f"  {nome:<20} [!] não encontrado")

In [ ]:
from pathlib import Path

from google.colab import files

# modelo.onnx e metadados.json são o que o site precisa
for arquivo in ["modelo.onnx", "metadados.json", "modelo.keras"]:
    if Path(arquivo).exists():
        print("Baixando:", arquivo)
        files.download(arquivo)
    else:
        print(f"[!] Não existe, pulando: {arquivo}")

---
# 10. Conclusões

## O que foi construído

Um classificador de resíduos recicláveis treinado por transfer learning sobre
MobileNetV2, exportado para ONNX e verificado contra o modelo original — pronto
para rodar no navegador sem servidor.

## As três decisões que definiram o projeto

**Transfer learning em duas fases.** Congelar a base primeiro não é
formalidade: a cabeça começa com pesos aleatórios, e seus gradientes iniciais
destruiriam as features do ImageNet se a base estivesse destravada. A ordem
importa mais que os hiperparâmetros.

**Normalização dentro do modelo.** Essa única escolha elimina a classe de bug
mais traiçoeira em visão computacional no navegador: o pré-processamento
divergente. Quando o JavaScript normaliza diferente do treino, nada quebra — o
modelo apenas passa a errar em silêncio. Com a conta dentro do grafo, o
contrato vira "mande pixels de 0 a 255" e não há como errar.

**Verificar a conversão numericamente.** Comparar Keras e ONNX imagem a imagem
custa cinco linhas e é a diferença entre saber que o modelo está correto e
torcer para que esteja.

## Por que classificação, e não detecção

Boa parte dos datasets de resíduo no Roboflow é de **detecção com caixas
delimitadoras**. A escolha por classificação foi deliberada, por duas razões.

A primeira é prática: detecção não se treina com Keras/TensorFlow de forma
razoável — exigiria outro framework.

A segunda é decisiva para o site. Modelos de detecção têm formatos de saída
incompatíveis entre si: YOLOv5 devolve `[1, 25200, 85]`, YOLOv8 devolve
`[1, 84, 8400]`, YOLO26 com cabeça end-to-end devolve `[1, 300, 6]`. Um site
que aceite "qualquer ONNX" não teria como adivinhar a decodificação correta. Em
classificação a saída é sempre `[1, n_classes]` com softmax — **universal**.

## Limitações honestas

- **Fotos de dataset não são fotos de celular.** As imagens do Roboflow tendem a
  ter o objeto centralizado, bem iluminado e em fundo limpo. No site, com foto
  de cozinha mal iluminada e três objetos na mesma cena, o desempenho cai.
- **O modelo não sabe dizer "não sei".** Uma foto de um gato vai receber alguma
  classe de resíduo com alta confiança. Tratar isso exigiria uma classe
  "nenhum" no treino ou um limiar mínimo de confiança no site.
- **Um objeto por imagem.** É classificação, não detecção: uma foto com garrafa
  e lata junta recebe um rótulo só.
- **As classes vêm do dataset escolhido.** Se ele não tiver "orgânico" ou
  "rejeito", o modelo nunca vai prever essas categorias.

## Próximo passo

Com `modelo.onnx` e `metadados.json` em mãos, o próximo passo é o site no
Lovable: carregar o modelo no navegador com `onnxruntime-web`, receber a foto
do usuário pela câmera ou por upload, e exibir as probabilidades. O
`metadados.json` foi desenhado justamente para que o site não precise de nenhum
valor escrito à mão — o que também permite aceitar o modelo de outra pessoa.